In [1]:
import pandas as pd 
df = pd.read_csv('LIAR_bal.csv') 

from sklearn.model_selection import train_test_split 
X_train_text, X_test_text, y_train, y_test = train_test_split( df['title'], df['label'], test_size=0.2, random_state=42 )

In [2]:
# =========================
# Fine-tune BERT
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))  # or len(set(df['label']))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)
model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6685


Training Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6056


...

Epoch 3 | Train Loss: 0.4559


Training Epoch 4:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.2811


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Validation Accuracy: 0.6244 | F1-score: 0.6242

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.65      0.64      1129
           1       0.62      0.60      0.61      1089

    accuracy                           0.62      2218
   macro avg       0.62      0.62      0.62      2218
weighted avg       0.62      0.62      0.62      2218


Confusion Matrix:
[[732 397]
 [436 653]]


In [3]:
# =========================
# Fine-tune RoBERTa
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6710


Training Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6318


...

Epoch 3 | Train Loss: 0.5491


Training Epoch 4:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.4200


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Validation Accuracy: 0.6474 | F1-score: 0.6447

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.73      0.68      1129
           1       0.67      0.56      0.61      1089

    accuracy                           0.65      2218
   macro avg       0.65      0.65      0.64      2218
weighted avg       0.65      0.65      0.64      2218


Confusion Matrix:
[[826 303]
 [479 610]]


In [4]:
import pickle

with open("best_transformer_model_roberta.pkl", "wb") as f:
    pickle.dump(model, f)

with open("roberta_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [5]:
# =========================
# Fine-tune ALBERT
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AlbertTokenizer,
    AlbertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')

model = AlbertForSequenceClassification.from_pretrained(
    'albert-base-v2',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6774


Training Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6465


...

Epoch 3 | Train Loss: 0.5921


Training Epoch 4:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 4 | Train Loss: 0.4379


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Validation Accuracy: 0.6326 | F1-score: 0.6325

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.62      0.63      1129
           1       0.62      0.65      0.63      1089

    accuracy                           0.63      2218
   macro avg       0.63      0.63      0.63      2218
weighted avg       0.63      0.63      0.63      2218


Confusion Matrix:
[[695 434]
 [381 708]]


In [6]:
# =========================
# Fine-tune RoBERTa
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

# ---------- Device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Number of labels ----------
num_labels = len(set(y_train))

# ---------- Dataset class ----------
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ---------- Load tokenizer & model ----------
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=num_labels
)

model.to(device)

# ---------- Datasets ----------
train_dataset = NewsDataset(
    X_train_text.tolist(),
    y_train.values if hasattr(y_train, "values") else y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test_text.tolist(),
    y_test.values if hasattr(y_test, "values") else y_test,
    tokenizer
)

# ---------- Dataloaders ----------
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# ---------- Optimizer & Scheduler ----------
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 5
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# ---------- Training ----------
for epoch in range(epochs):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):

        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss / len(train_loader):.4f}")

# ---------- Evaluation ----------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# ---------- Metrics ----------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\nValidation Accuracy: {acc:.4f} | F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(cm)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Epoch 1:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6732


Training Epoch 2:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.6249


...

Epoch 4 | Train Loss: 0.3766


Training Epoch 5:   0%|          | 0/555 [00:00<?, ?it/s]

Epoch 5 | Train Loss: 0.2298


Evaluating:   0%|          | 0/139 [00:00<?, ?it/s]


Validation Accuracy: 0.6258 | F1-score: 0.6244

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.68      0.65      1129
           1       0.63      0.57      0.60      1089

    accuracy                           0.63      2218
   macro avg       0.63      0.62      0.62      2218
weighted avg       0.63      0.63      0.62      2218


Confusion Matrix:
[[771 358]
 [472 617]]


In [1]:
import pickle

with open("best_transformer_model_roberta.pkl", "rb") as f:
    model = pickle.load(f)

with open("roberta_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

In [2]:
model.save_pretrained("roberta_finetuned_model")
tokenizer.save_pretrained("roberta_finetuned_model")

('roberta_finetuned_model\\tokenizer_config.json',
 'roberta_finetuned_model\\special_tokens_map.json',
 'roberta_finetuned_model\\vocab.json',
 'roberta_finetuned_model\\merges.txt',
 'roberta_finetuned_model\\added_tokens.json')

In [3]:
import pickle
with open("roberta_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

In [4]:
tokenizer.save_pretrained("tokenizer_finetuned_model")

('tokenizer_finetuned_model\\tokenizer_config.json',
 'tokenizer_finetuned_model\\special_tokens_map.json',
 'tokenizer_finetuned_model\\vocab.json',
 'tokenizer_finetuned_model\\merges.txt',
 'tokenizer_finetuned_model\\added_tokens.json')

In [5]:
from transformers import RobertaModel
roberta_model = RobertaModel.from_pretrained("roberta-base")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
roberta_model.save_pretrained("roberta_model")